In [ ]:
# Preparar datos de ejemplo:

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Generar datos de ventas
np.random.seed(42)
ventas = pd.DataFrame({
    'venta_id': range(1, 1001),
    'cliente_id': np.random.randint(1, 101, 1000),
    'producto_id': np.random.randint(1, 51, 1000),
    'cantidad': np.random.randint(1, 11, 1000),
    'precio_unitario': np.round(np.random.uniform(10, 500, 1000), 2),
    'fecha_venta': pd.date_range('2024-01-01', periods=1000, freq='1H'),
    'updated_at': datetime.now()
})

ventas['total'] = ventas['cantidad'] * ventas['precio_unitario']

print(f"Generados {len(ventas)} registros de ventas")
print(ventas.head())

Generados 1000 registros de ventas
   venta_id  cliente_id  producto_id  cantidad  precio_unitario  \
0         1          52           34         4           405.14   
1         2          93           47        10           235.03   
2         3          15            8         6            35.46   
3         4          72           40         7           395.28   
4         5          61           49         2           108.67   

          fecha_venta                 updated_at    total  
0 2024-01-01 00:00:00 2026-01-15 17:46:04.874625  1620.56  
1 2024-01-01 01:00:00 2026-01-15 17:46:04.874625  2350.30  
2 2024-01-01 02:00:00 2026-01-15 17:46:04.874625   212.76  
3 2024-01-01 03:00:00 2026-01-15 17:46:04.874625  2766.96  
4 2024-01-01 04:00:00 2026-01-15 17:46:04.874625   217.34  


In [ ]:
# Carga completa (carga completa):

In [2]:
import sqlite3

def carga_completa_sqlite(df, tabla):
    conn = sqlite3.connect(':memory:')
    
    # Crear tabla
    conn.execute(f'''
        CREATE TABLE {tabla} (
            venta_id INTEGER PRIMARY KEY,
            cliente_id INTEGER,
            producto_id INTEGER,
            cantidad INTEGER,
            precio_unitario REAL,
            total REAL,
            fecha_venta TEXT,
            updated_at TEXT
        )
    ''')
    
    # Insertar datos
    df.to_sql(tabla, conn, if_exists='replace', index=False)
    
    # Verificar
    cursor = conn.execute(f"SELECT COUNT(*) FROM {tabla}")
    count = cursor.fetchone()[0]
    
    conn.close()
    return count

registros_cargados = carga_completa_sqlite(ventas, 'ventas_completas')
print(f"Carga completa: {registros_cargados} registros")

Carga completa: 1000 registros


In [ ]:
# Carga incremental (simulada):

In [3]:
def carga_incremental(df, archivo_parquet, ultimo_id=0):
    # Simular carga incremental: solo registros nuevos
    nuevos_registros = df[df['venta_id'] > ultimo_id]
    
    if len(nuevos_registros) > 0:
        try:
            # En producción, leer archivo existente y append
            nuevos_registros.to_parquet(
                archivo_parquet,
                engine='pyarrow',
                index=False
            )
            print(f"Carga incremental: {len(nuevos_registros)} nuevos registros")
            return len(nuevos_registros)
        except Exception as e:
            print(f"Error en carga incremental: {e}")
            return 0
    else:
        print("No hay nuevos registros para cargar")
        return 0

nuevos_cargados = carga_incremental(ventas, 'ventas_incremental.parquet', ultimo_id=500)
print(f"Registros nuevos agregados: {nuevos_cargados}")

Carga incremental: 500 nuevos registros
Registros nuevos agregados: 500


In [4]:
print(ventas.head())

   venta_id  cliente_id  producto_id  cantidad  precio_unitario  \
0         1          52           34         4           405.14   
1         2          93           47        10           235.03   
2         3          15            8         6            35.46   
3         4          72           40         7           395.28   
4         5          61           49         2           108.67   

          fecha_venta                 updated_at    total  
0 2024-01-01 00:00:00 2026-01-15 17:46:04.874625  1620.56  
1 2024-01-01 01:00:00 2026-01-15 17:46:04.874625  2350.30  
2 2024-01-01 02:00:00 2026-01-15 17:46:04.874625   212.76  
3 2024-01-01 03:00:00 2026-01-15 17:46:04.874625  2766.96  
4 2024-01-01 04:00:00 2026-01-15 17:46:04.874625   217.34  


In [5]:
print(ventas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   venta_id         1000 non-null   int64         
 1   cliente_id       1000 non-null   int32         
 2   producto_id      1000 non-null   int32         
 3   cantidad         1000 non-null   int32         
 4   precio_unitario  1000 non-null   float64       
 5   fecha_venta      1000 non-null   datetime64[ns]
 6   updated_at       1000 non-null   datetime64[us]
 7   total            1000 non-null   float64       
dtypes: datetime64[ns](1), datetime64[us](1), float64(2), int32(3), int64(1)
memory usage: 50.9 KB
None


In [ ]:
# Comparar estrategias:

In [9]:
import time

def comparar_estrategias_carga():
    estrategias = {}
    
    # Medir carga completa
    start = time.time()
    carga_completa_sqlite(ventas, 'ventas_test')
    estrategias['completa'] = time.time() - start
    
    # Medir carga incremental (simulada)
    start = time.time()
    carga_incremental(ventas, 'ventas_inc_test.parquet', ultimo_id=800)
    estrategias['incremental'] = time.time() - start
    
    print("Comparación de estrategias:")
    print(".2f")
    print(".2f")
    
    return estrategias

resultados = comparar_estrategias_carga()

Carga incremental: 200 nuevos registros
Comparación de estrategias:
.2f
.2f


In [10]:
import time

def comparar_estrategias_carga():
    estrategias = {}
    
    # Medir carga completa
    start = time.time()
    carga_completa_sqlite(ventas, 'ventas_test')
    estrategias['completa'] = time.time() - start
    
    # Medir carga incremental (simulada)
    start = time.time()
    carga_incremental(ventas, 'ventas_inc_test.parquet', ultimo_id=800)
    estrategias['incremental'] = time.time() - start
    
    print("Comparación de estrategias:")
    print(f"Carga completa:     {estrategias['completa']:.2f} segundos")
    print(f"Carga incremental:  {estrategias['incremental']:.2f} segundos")
    
    return estrategias

resultados = comparar_estrategias_carga()

Carga incremental: 200 nuevos registros
Comparación de estrategias:
Carga completa:     0.03 segundos
Carga incremental:  0.01 segundos


In [ ]:
""" 
¿En qué situaciones usarías carga completa vs incremental? 

Carga Completa, se usa cuando:

    • El dataset es pequeño o moderado y la carga completa no afecta tiempos ni costos.
    • La calidad de los datos es incierta y se necesita “resetear” la tabla destino.
    • Hay riesgo de drift o corrupción histórica, y se quiere reconstruir desde cero.
    • No existen claves primarias o marcas de cambio (timestamps, updated_at, IDs crecientes).
    • El modelo cambia (nuevas columnas, cambios de tipo, nuevas reglas de negocio).
    • Procesos de auditoría requieren reconstrucción periódica (por ejemplo, mensual).

    Ventaja clave: simplicidad y consistencia total.
    Desventaja: poco escalable cuando el volumen crece.

Carga Incremental, se usa cuando:

    • El volumen es grande y recargar todo sería costoso.
    • Los datos cambian parcialmente (solo nuevos o modificados).
    • Se tienes claves primarias confiables o marcas de cambio (timestamps, versioning, CDC).
    • Se necesitan cargas frecuentes (cada hora, cada 5 minutos, streaming).
    • El SLA exige baja latencia.
    • El sistema origen soporta CDC (Change Data Capture, logs binarios, Debezium, etc.).

    Ventaja clave: eficiencia en tiempo, cómputo y costos.
    Desventaja: mayor complejidad y riesgo si no se controla bien la lógica de cambios.


¿Qué factores influyen en el tamaño óptimo de batch para carga de datos?

    El tamaño del batch es un equilibrio entre velocidad, estabilidad y costo.
    Estos son los factores que importan y a tener en cuenta:

    1. Capacidad del sistema origen
        • Límites de concurrencia
        • Sensibilidad a cargas pesadas
        • Ventanas de mantenimiento
        Si el origen es frágil, batches pequeños reducen impacto.

    2. Capacidad del sistema destino
        • Velocidad de escritura
        • Índices y constraints
        • Contención de locks
        Bases como PostgreSQL o MySQL pueden degradarse con batches gigantes.

    3. Ancho de banda y latencia
        • Si la red es lenta, conviene enviar menos batches pero más grandes.
        • Si la red es rápida, batches pequeños funcionan bien.

    4. Tamaño de cada registro
        No es lo mismo batch de 10.000 filas de 20 bytes que 10.000 filas de 5 KB.

    5. Estrategia de transacciones
        • Batches grandes implican menos commits, más riesgo si falla.
        • Batches pequeños implican más commits, más overhead.

    6. Costos de cómputo
        En cloud, el batch óptimo suele minimizar:
        • tiempo de ejecución
        • I/O
        • reintentos

    7. Frecuencia de actualización
        • Si la carga es cada 5 minutos, batches pequeños.
        • Si la cargas una vez al día, batches más grandes.
"""